# 07 · Choose a model, check its answer

**General question → Switchyard → Lightning base**

**Classification → Switchyard → Lightning specialist**

These routes are selected explicitly by the tool. Switchyard resolves the named route; it does not infer the task from the question in this example.

**Who does what:** Nano orchestrates the lesson tools. Switchyard routes their requests to Lightning base or specialist. Application code validates the returned classification.

###  explicit routing and application contracts

the [Switchyard quickstart](https://nvidia-nemo.github.io/Switchyard/getting_started/) explains routing. The workshop pins `nemo-switchyard==0.3.0`; the current guide may show newer interfaces. Our exact configuration is [switchyard.toml](../run/workshop/configs/switchyard.toml).

start the Lightning endpoint with both aliases, then run the route, gate, and ordering checks. [routing_trace.py](agent/routing_trace.py) creates a temporary embedded router; you do not start a separate Switchyard daemon.

**What is enforced:** `workshop-reason` targets `base`; `workshop-triage` targets `specialist`. [contracts.py](../run/workshop/workshop_agent/contracts.py) checks labels/confidence and assigns the destination by lookup. [runtime.py](agent/runtime.py) blocks drafting before evidence and valid classification.

these are fixed routes and deterministic gates. Switchyard does not learn model selection here, and a corrected destination is application behavior. Inspect [the lesson source](agent/lessons/07-specialist-tool.json) to distinguish model output from code-enforced decisions.


### Start here
Run the short cells in order. Helpers execute the existing lesson code and keep detailed logs expandable. All displayed results come from your run. Starting again creates a fresh output folder.

In [ ]:
from pathlib import Path
import sys, json
from IPython.display import JSON, display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "course/common.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from course.agent.lesson_walkthrough import LessonWalkthrough, table, bars, loss_chart, show_skill_library
lesson = LessonWalkthrough('07-specialist-tool')

### Configuration pattern · one route has one target

This excerpt is from our tested [Switchyard configuration](../run/workshop/configs/switchyard.toml). Read it alongside the [upstream quickstart](https://nvidia-nemo.github.io/Switchyard/getting_started/); it is not an adaptive classifier.

```toml
[routes.triage]
id = "workshop-triage"
type = "passthrough"
target = "specialist"
```

The wrapper sends `workshop-triage` to the router, then checks that the response identifies `specialist`. A mismatch is an error, not a silent fallback.


### 1 · Run both routes

Ask one general question and classify one recorded observation. These are fresh HTTP calls through Switchyard. Read the returned model IDs and timings.

In [ ]:
lesson.step("route")
table(["Route", "Actual model", "Finish", "Seconds", "Answer"], [(r["route"], r["model"], r["finish_reason"], r["seconds"], r["answer"]) for r in lesson.ns["routing_results"]])

### 2 · Make the rules visible

| Proposal | Application behavior |
|---|---|
| Civil gathering with wrong destination | Route to CIMIC using the lookup table |
| One source claiming CONFIRMED | Reject |

These are deliberate test inputs. Passing a contract does not prove the scene was understood correctly.

In [ ]:
lesson.step("gate")
check = lesson.ns["validate_item"]
table(["Test", "Actual gate result"], [("Destination correction", check(lesson.ns["valid"],1)), ("Overconfidence rejection", check(lesson.ns["invalid"],1))])

### 3 · Try a tool too early

**No evidence → request draft → rejection**

The application enforces preconditions even if a model asks for the wrong tool. Run this intentional failure and inspect the rejection in the execution details.

In [ ]:
lesson.step("order")
print("Allowed at the start:", lesson.ns["allowed_actions"](lesson.ns["state"], "draft-report"))

### Check and save this manual run
Validate the artifacts and record this run for the next manual walkthrough. A successful check describes execution and evidence; read the actual quality scores separately.

In [ ]:
manual_result = lesson.finish()
display(JSON(manual_result, expanded=False))

### Grow the skill library
The library now includes this lesson’s skill, **workshop-specialist-tool**. Open the actual instructions below.

**`inspect-tool` → `test-tool` → `check-tool`**

In [ ]:
show_skill_library('07-specialist-tool')

### Prepare OpenClaw
Run once for each new demo. This cell installs the growing skill library in a fresh sandbox workspace and starts a background host worker for this lesson’s three actions. **It does not ask the agent to do anything yet.**

Here, **installing a skill means placing its `SKILL.md` instruction file in OpenClaw’s workspace**. The current lesson’s source is `course/agent/skills/workshop-specialist-tool/SKILL.md` in this repository.

Each run gets this structure inside the OpenShell sandbox (launcher files omitted):

```text
/home/sandbox/course-runs/openclaw-<run-id>/
├── openclaw.json                 # points OpenClaw at this run’s workspace
├── chapter_tool.py              # helper invoked by the agent
├── chapter.json                 # this lesson’s task and allowed actions
└── workspace/
    ├── AGENTS.md                 # workspace instructions
    └── skills/
        ├── workshop-specialist-tool/
        │   └── SKILL.md
        └── ...                  # skills from earlier notebooks
```

OpenClaw discovers skills under `workspace/skills/`. The task you paste in the next step tells the agent which skill to read and supplies the full path to `chapter_tool.py` (the skill’s `<tool_path>` placeholder).

The agent invokes the helper through its execution tool. The helper forwards the selected action to the host worker, which runs the fixed lesson code and returns the result. Nano follows the skill and coordinates the calls.

This lesson does not require artifacts from an earlier OpenClaw demo.

Start the TUI task within **15 minutes** and keep this run’s `terminal_run` value for collecting results. If the worker expires, prepare a new run. The local run directory stores host results; the installed skills live at the sandbox paths printed below.


In [ ]:
from course.agent.chapter_terminal import prepare_chapter_terminal, collect_chapter_terminal
from course.agent.notebook_terminal import open_terminal
terminal_run = prepare_chapter_terminal('07-specialist-tool')
print("Local run directory:", terminal_run["directory"])
print("OpenClaw sandbox workspace:", terminal_run["remote"] + "/workspace")
print("Installed skill:", terminal_run["remote"] + "/workspace/skills/workshop-specialist-tool/SKILL.md")
print("Python helper:", terminal_run["remote"] + "/chapter_tool.py")
print("\nThe agent will run these commands individually:")
for action in ['inspect-tool', 'test-tool', 'check-tool']:
    print("python3 " + terminal_run["remote"] + "/chapter_tool.py " + action)

### Open the TUI and paste the task
Exit the previous TUI with `/exit` first. This cell opens a terminal on the workshop computer’s desktop. Copy only the generated task into OpenClaw.

The terminal label is **Nano**, its lightweight orchestration model. Nano orchestrates the lesson tools. Switchyard routes their requests to Lightning base or specialist. Application code validates the returned classification.

In [ ]:
open_terminal(terminal_run)
print("\nPASTE THIS INTO OPENCLAW:\n")
print(terminal_run["prompt"])

### Bring the evidence back
After the TUI finishes, run this cell. It retrieves that session’s trace and checks the artifacts and skill reads. If Run All reaches this cell before you finish the TUI task, run it again afterward.

**Look for:** General → base; classification → specialist; actual model IDs checked.

In [ ]:
agent_result = collect_chapter_terminal(terminal_run)
display(JSON(agent_result, expanded=True))

**What you demonstrated:** The model proposes an answer or action. Code enforces the contract and order.

Exit the TUI with `/exit` when done.